# 03 — Train a DQN Model Online

Train a MOUSE model directly against live `mouse-env` environments while keeping replay data in memory.

The online loop has three jobs running together:

1. **Collect fresh experience.** The current model chooses actions in live env instances with epsilon-greedy exploration. Each `(action, observation, reward, done, ...)` row is appended to a `Datastore`.
2. **Sample replay batches.** The `Datastore`s act as an in-memory replay buffer. A `DataLoader` samples contiguous sequence windows from those stores and returns `[batch][sequence]` raw dict batches.
3. **Update the model.** The model trains on replay batches with `DqnObjective`, then Polyak-updates the target head.

A small per-env policy context is still kept for choosing the next live action. Training, however, always goes through `DataLoader.next_batch()` so collection and replay updates use one batch interface.

In [ ]:
import torch

from mouse_envs import EnvConfig, make_env
from mouse_core.data import DataLoader, Datastore
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import StepEmbedder
from mouse_core.models.heads import DiscreteActionValueHead
from mouse_core.objectives import DqnObjective

MODEL_ID = "mouse-example-model"
MAX_ACTIONS = 4
MAX_OBS_DISCRETE = 64
NUM_ENVS = 1000
SEQUENCE_LENGTH = 128
BATCH_SIZE = 4
TRAIN_STEPS = 4000
LEARNING_STARTS = SEQUENCE_LENGTH
EPSILON_START = 1.0
EPSILON_END = 0.05


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Build Environment

Online training keeps the environment in the training loop. Each env instance runs one infinite task with deterministic FrozenLake dynamics and 50-step episode truncation, so the model keeps carrying policy context across episode boundaries.

The env returns plain step dictionaries with fields such as `observation`, `reward`, and `done`. We merge each output with the input action that produced it, which gives replay one row per environment step.

In [ ]:
configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"proc_frozenlake_online_{i}",
        reset_seed=i,
        episodes_per_task=0,
        kwargs={
            "is_slippery": False,
            "min_width": 4, "max_width": 8,
            "min_height": 4, "max_height": 8,
            "step_penalty": -0.01,
            "max_episode_steps": 50,
            "map_seed": i,
        },
    )
    for i in range(NUM_ENVS)
]

env = make_env(configs)
print("Environments:", env.names)

Environments: ('proc_frozenlake_online_0', 'proc_frozenlake_online_1', 'proc_frozenlake_online_2', 'proc_frozenlake_online_3')


## Build the model

The model uses a DQN architecture: a `StepEmbedder`, the full pretrained `Qwen3Backbone`, and a `DiscreteActionValueHead`.

For faster debugging you can pass `num_layers=...` to truncate the backbone, or lower `TRAIN_STEPS`. This tuned training example uses all pretrained backbone layers.

In [3]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B")

encoder = StepEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "reward",
            "type": "rff",
            "std": 0.02,
            "in_min": 0.01,
            "in_max": 100.0,
            "tokens": 1,
        },
        {
            "field": "done",
            "type": "discrete",
            "vocab_size": 5,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "type": "learnable",
            "tokens": 1,
        },
    ],
    concat_modalities=False,
    include_type_token=False,
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=1.0,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

ValueError: in_min and in_max must be > 0

## Online replay helpers

Each env instance writes to one `Datastore`. Together, those stores are the replay buffer. New rows are appended every environment step, then a `DataLoader` samples random contiguous windows from the current stores.

`DataLoader` snapshots stores when it is constructed, so this example rebuilds a synchronous loader after appending new rows. That keeps the example simple and makes the replay update explicit: collect new rows, refresh the loader view, sample a batch.

`policy_contexts[i]` is separate from replay. It holds the task history needed to choose the next live action for env instance `i`. This example uses `episodes_per_task=0`, so that context normally carries for the whole run; the task-boundary clear is still present for configs that emit done code `3` or `4`.

In [ ]:
def epsilon_for_step(step: int) -> float:
    progress = min(step / max(TRAIN_STEPS - 1, 1), 1.0)
    return EPSILON_START + progress * (EPSILON_END - EPSILON_START)


def is_task_done(row: dict) -> bool:
    done = row.get("done", 0)
    if isinstance(done, torch.Tensor):
        done = done.item()
    return int(done) in {3, 4}


def choose_inputs(policy_contexts: list[list[dict]], epsilon: float) -> list[dict]:
    random_inputs = env.sample_random_inputs()
    inputs = []

    for i, context in enumerate(policy_contexts):
        if not context or torch.rand(1).item() < epsilon:
            inputs.append(random_inputs[i])
            continue

        with torch.no_grad():
            predictions, _, _ = model([context[-SEQUENCE_LENGTH:]])
            action = model.get_action(predictions, temperature=0.0, num_actions=MAX_ACTIONS)
        inputs.append({"action": action.squeeze().cpu()})

    return inputs


def append_to_replay(
    stores: list[Datastore],
    policy_contexts: list[list[dict]],
    inputs: list[dict],
    outputs: list[dict],
) -> None:
    for store, context, inp, out in zip(stores, policy_contexts, inputs, outputs):
        row = {**inp, **out}
        store.append(row)
        context.append(row)
        if is_task_done(row):
            context.clear()


def replay_ready(stores: list[Datastore]) -> bool:
    return all(len(store) >= SEQUENCE_LENGTH for store in stores)


def make_replay_loader(stores: list[Datastore]) -> DataLoader:
    return DataLoader(
        stores,
        sequence_length=SEQUENCE_LENGTH,
        batch_size=BATCH_SIZE,
        weight_mode="per_step",
        num_workers=0,
    )

## Train online

Each iteration does the full online cycle:

1. Compute the current exploration rate.
2. Choose one action per env from the model's current policy context, falling back to random actions during exploration or before any context exists.
3. Step the live environments.
4. Append each merged row to its env's `Datastore`, growing the replay buffer.
5. Once every store has at least `SEQUENCE_LENGTH` rows, refresh the `DataLoader` view of replay and sample a batch with `next_batch()`.
6. Run the same model/objective/optimizer update used by offline training.

The first env step triggers the initial reset frame, so the loop collects replay rows before the first gradient update.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1.0e-5, weight_decay=0.0, betas=(0.9, 0.95), eps=1.0e-8)
objective = DqnObjective(
    gamma_step=1.0,
    gamma_episode_terminal=0.99,   # bootstrap through episode boundaries
    gamma_episode_truncated=0.99,
    gamma_task_terminal=0.99,      # bootstrap through task boundaries too
    gamma_task_truncated=0.99,
    tau=0.0005,
)

stores = [Datastore(name=name) for name in env.names]
policy_contexts = [[] for _ in env.names]
env.tracker.clear()

for step in range(TRAIN_STEPS):
    epsilon = epsilon_for_step(step)
    inputs = choose_inputs(policy_contexts, epsilon=epsilon)
    outputs = env.step(inputs)
    append_to_replay(stores, policy_contexts, inputs, outputs)

    if step >= LEARNING_STARTS and replay_ready(stores):
        loader = make_replay_loader(stores)
        try:
            batch = loader.next_batch()
        finally:
            loader.close()

        predictions, objective_data, _ = model(batch)
        loss, metrics = objective(objective_data, predictions)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        model.polyak_update(action_value_tau=objective.tau)

        if step % 100 == 0:
            replay_steps = sum(len(store) for store in stores)
            print(
                f"step={step} epsilon={epsilon:.3f} replay_steps={replay_steps} "
                f"loss={loss.item():.4f} q={metrics['q_values_mean']:.3f}"
            )

env.close()
print("Online training finished.")

for name, rewards, lengths in zip(env.names, env.tracker.episode_cum_rewards, env.tracker.episode_lengths):
    n = len(rewards)
    mean_r = torch.tensor(rewards).mean().item() if n else float("nan")
    mean_l = torch.tensor(lengths).float().mean().item() if n else float("nan")
    print(f"{name}: {n} episodes | mean reward {mean_r:.3f} | mean length {mean_l:.1f}")

## Push to the Hub

Run this cell if you want to save the online-trained checkpoint to the shared Hub repo under `MODEL_ID`. The inference notebook loads the current checkpoint from that repo, so whichever training notebook you push last is the model it evaluates.

In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model, MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")